# 05 — Full Pipeline End-to-End Test

Goal: Run the complete anime-to-live-action pipeline through the Deep Agents coordinator.

Prerequisites:
- All previous notebooks pass (LM Studio or Gemini available)
- Ollama running (default) OR another model configured in `.env`

This test uses the existing mock tool implementations. The goal is to validate:
1. The coordinator delegates to all 7 subagents in order
2. Each subagent calls its tools and returns structured output
3. The quality review loop works (or is skipped if score >= 0.7)
4. The final assembly step produces a result

In [ ]:
import os, sys
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.insert(0, "..")

# Use ollama by default for testing
os.environ.setdefault("AGENT_MODEL", "ollama:llama3")
os.environ.setdefault("DEFAULT_MODE", "hybrid")
os.environ.setdefault("MAX_PIPELINE_ITERATIONS", "3")

print(f"Agent model: {os.environ['AGENT_MODEL']}")
print(f"Mode: {os.environ['DEFAULT_MODE']}")
print(f"Max iterations: {os.environ['MAX_PIPELINE_ITERATIONS']}")

## 5.1 Create the pipeline agent

In [ ]:
from app.agents.pipeline import create_pipeline_agent
from app.agents.state import PipelineState

try:
    pipeline = create_pipeline_agent()
    print(f"Pipeline agent created: {type(pipeline).__name__}")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback; traceback.print_exc()
    raise

## 5.2 Invoke the pipeline

In [ ]:
import time
start = time.time()

try:
    result = pipeline.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "Run the anime-to-live-action pipeline on /test/anime.mp4. "
                        "Mode: hybrid. Max iterations: 3."
                    ),
                }
            ],
        },
        config={"configurable": {"thread_id": "e2e-test-01"}},
    )
    elapsed = time.time() - start
    print(f"Pipeline completed in {elapsed:.1f}s")
except Exception as e:
    elapsed = time.time() - start
    print(f"Pipeline FAILED after {elapsed:.1f}s")
    import traceback; traceback.print_exc()

## 5.3 Inspect results

In [ ]:
# Show the final message content
messages = result.get("messages", [])
print(f"Total messages: {len(messages)}")

for i, msg in enumerate(messages):
    role = msg.type if hasattr(msg, 'type') else type(msg).__name__
    content = str(msg.content)
    print(f"\n--- Message {i} ({role}) ---")
    print(content[:500])
    if len(content) > 500:
        print(f"... (truncated, {len(content)} total chars)")

## 5.4 Verify pipeline state

If the pipeline uses `PipelineState`, check which steps produced data.

In [ ]:
# Check if state was populated
state_keys = [k for k in result.keys() if not k.startswith("_")]
print("Result keys:", state_keys)

# If PipelineState fields are present, show them
for key in ["analysis_result", "character_designs", "reference_images", 
            "shot_plan", "generated_clips", "quality_review", "final_video_path"]:
    val = result.get(key)
    if val is not None:
        print(f"  {key}: {str(val)[:200]}")

## 5.5 Test quality loop behavior

This validates that when quality score is below threshold, the coordinator loops back.

In [ ]:
# Run with low quality threshold to test the loop
result2 = pipeline.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Run the anime-to-live-action pipeline on /test/anime2.mp4. "
                    "Mode: local. Max iterations: 2."
                ),
            }
        ],
    },
    config={"configurable": {"thread_id": "e2e-test-loop"}},
)

messages2 = result2.get("messages", [])
print(f"Total messages for loop test: {len(messages2)}")
print(f"Last message: {str(messages2[-1].content)[:300]}" if messages2 else "No messages")

## 5.6 Test FastAPI endpoint directly

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

# Health check
resp = client.get("/health")
print(f"Health: {resp.status_code} — {resp.json()}")

# Generate endpoint
resp = client.post(
    "/generate",
    json={"clip_path": "/test/test.mp4", "mode": "hybrid", "max_iterations": 1},
)
print(f"Generate: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    result_content = data.get("result", "")[:300]
    print(f"Result: {result_content}")
else:
    print(f"Error: {resp.text[:500]}")

## Summary

| Check | Status |
|-------|--------|
| Pipeline agent created | ⬜ |
| Full pipeline invocation | ⬜ |
| All 7 steps executed | ⬜ |
| Quality loop works | ⬜ |
| FastAPI endpoints work | ⬜ |

If the pipeline fails, check:
- The LLM backend is running (ollama, LM Studio, etc.)
- The model name matches what's available
- All tool modules are importable without errors